In [14]:
# =========================================================================
# CÉLULA 1: CARREGAMENTO DE DADOS (USANDO CAMINHO ABSOLUTO)
# =========================================================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, f1_score, accuracy_score, confusion_matrix

print("--- FASE 1: CARREGAMENTO ---")

# ATENÇÃO: Se usar o Colab, este caminho NÃO FUNCIONARÁ.
# Este caminho é para o Jupyter Notebook/ambiente local no seu PC.
BASE_PATH = "C:\\Users\\User\\Desktop\\escola\\Projeto\\PlaySafe4All\\PlaySafe4All\\Data"

# Construindo os caminhos completos
FILE_PATH_1 = f"{BASE_PATH}\\Base Artigo 1.csv"
FILE_PATH_2 = f"{BASE_PATH}\\Base Artigo 2.csv"
FILE_PATH_3 = f"{BASE_PATH}\\Base Artigo 3.csv"

# Inicializar os dataframes
df1, df2, df3 = None, None, None
load_successful = False

# Carregar os três arquivos CSV
try:
    # O r antes das aspas é para tratar as barras invertidas do Windows
    df1 = pd.read_csv(FILE_PATH_1, sep=',')
    df2 = pd.read_csv(FILE_PATH_2, sep=',')
    df3 = pd.read_csv(FILE_PATH_3, sep=',')
    print("Arquivos carregados com sucesso.")
    load_successful = True
except FileNotFoundError:
    print("ERRO CRÍTICO: Não foi possível encontrar os ficheiros CSV. Verifique se o caminho completo no código está correto.")
    
# Mostrar informações básicas (apenas se o carregamento foi bem sucedido)
if load_successful:
    print(f"Total de amostras brutas: {len(df1) + len(df2) + len(df3)}")
    print(f"Artigo 1 shape: {df1.shape}") 
    print(f"Artigo 2 shape: {df2.shape}") 
    print(f"Artigo 3 shape: {df3.shape}")

--- FASE 1: CARREGAMENTO ---
Arquivos carregados com sucesso.
Total de amostras brutas: 181
Artigo 1 shape: (60, 233)
Artigo 2 shape: (61, 173)
Artigo 3 shape: (60, 341)


In [24]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

# AVISO: Assume que df1, df2 e df3 foram carregados na Célula 1!

print("\n--- FASE 2: PRÉ-PROCESSAMENTO E SELEÇÃO DE FEATURES (COMPLETA E 50/50) ---")

# 1. UNIÃO E FILTRAGEM DE FEATURES
df_combined = pd.concat([df1, df2, df3], ignore_index=True)

if 'Numero' in df_combined.columns:
    df_combined.drop(columns=['Numero'], inplace=True)

TARGET_COL = 'Entorse'

# LISTA COMPLETA DE FEATURES SELECIONADAS (Modelo mais robusto)
COLUNAS_SELECIONADAS = [
    TARGET_COL,             
    #'Posição',              # DESCOMENTADO
    #'M.I.Dominante',        # DESCOMENTADO
    #'Lado_entorse',         # DESCOMENTADO
    #'T0IMC',                # DESCOMENTADO
    #'T0MassaGordaPerc',     # DESCOMENTADO
    #'T0MassaMagra',         # DESCOMENTADO
    # Fatores de Exposição/Carga (Carga Aguda)
    'T0_T1_Match_Time_exposure', 
    'T0_T1_Training_Time_exposure',
    # Fatores de Risco Biomecânico (Performance)
    'T0SRTMax',             # Flexibilidade
    'T0TTestMin',           # Agilidade
    'T0SJmMax',             # Força Explosiva
    'T0Veli',               # Velocidade (30m)
    'T0RazaoAADto',         # Rácio Isocinético Direito
    'T0RazaoAAEsq',         # Rácio Isocinético Esquerdo
]

# Filtra o DataFrame
cols_to_keep = list(set(COLUNAS_SELECIONADAS) & set(df_combined.columns))
df_filtered = df_combined[cols_to_keep].copy()


# 2. TRATAMENTO DE VALORES OMISSOS E MAPEAMENTO DA CLASSE ALVO
# --- CORREÇÃO DO ERRO 'got [1 2]' DE EXECUÇÕES ANTERIORES ---
df_filtered[TARGET_COL].fillna(1, inplace=True) # Preenche NaN com o valor da classe negativa (1)
# Mapeamento: 2 (Entorse) -> 1 (Classe Positiva); 1 (Não Entorse) -> 0 (Classe Negativa)
df_filtered[TARGET_COL] = df_filtered[TARGET_COL].replace({2: 1, 1: 0})
df_filtered[TARGET_COL] = df_filtered[TARGET_COL].astype(int)
# -----------------------------------------------------------


# Identifica colunas para imputação e codificação
numerical_cols = df_filtered.select_dtypes(include=np.number).columns.tolist()
if TARGET_COL in numerical_cols:
    numerical_cols.remove(TARGET_COL) 
categorical_cols = df_filtered.select_dtypes(include=['object']).columns.tolist()

# Imputação de NaN (Median para numéricas, Mode para categóricas)
for col in numerical_cols:
    df_filtered[col] = df_filtered[col].fillna(df_filtered[col].median())

for col in categorical_cols:
    df_filtered[col] = df_filtered[col].fillna(df_filtered[col].mode()[0])

# 3. DEFINIÇÃO DE X/Y E CODIFICAÇÃO CATEGÓRICA
Y = df_filtered[TARGET_COL] 
X = df_filtered.drop(columns=[TARGET_COL]) 
X = pd.get_dummies(X, drop_first=True) 

print(f"Dimensão de X após codificação: {X.shape}") 


# 4. DIVISÃO TREINO/TESTE E NORMALIZAÇÃO
# MUDANÇA: Divisão 50/50 para comparação justa e robusta
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.75, random_state=42, stratify=Y)

# Normalização
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print(f"Dimensão de Treino (X_train): {X_train.shape[0]} amostras.")
print(f"Dimensão de Teste (X_test): {X_test.shape[0]} amostras.")
print("--- ✅ FASE 2 AJUSTADA! DADOS PRONTOS PARA XGBOOST. ---")


--- FASE 2: PRÉ-PROCESSAMENTO E SELEÇÃO DE FEATURES (COMPLETA E 50/50) ---
Dimensão de X após codificação: (181, 56)
Dimensão de Treino (X_train): 45 amostras.
Dimensão de Teste (X_test): 136 amostras.
--- ✅ FASE 2 AJUSTADA! DADOS PRONTOS PARA XGBOOST. ---


C:\Users\User\AppData\Local\Temp\ipykernel_14648\3026878245.py:46: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_filtered[TARGET_COL].fillna(1, inplace=True) # Preenche NaN com o valor da classe negativa (1)


In [25]:
from xgboost import XGBClassifier
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore', category=UserWarning) # Ignora alguns warnings do XGBoost

print("\n--- FASE 3: TREINO DO MODELO (XGBoost) ---")

# 1. Cálculo da Ponderação de Classes (Para o desequilíbrio)
# O XGBoost precisa do rácio (Nº Negativos / Nº Positivos)
total_samples = len(y_train)
positive_samples = y_train.sum()  # Entorse (Y=1)
negative_samples = total_samples - positive_samples  # Não Entorse (Y=0)
scale_ratio = negative_samples / positive_samples if positive_samples > 0 else 1 

# 2. Inicialização do XGBoost
xgb_model = XGBClassifier(
    n_estimators=150, 
    max_depth=4,      
    learning_rate=0.1, 
    random_state=42,
    use_label_encoder=False, 
    eval_metric='logloss',
    scale_pos_weight=scale_ratio  
)

# 3. Treino
# --- CORREÇÃO DE ERRO: Conversão explícita para Numpy (.values) ---
# Isso garante que o XGBoost receba apenas dados numéricos e evita o erro 'DataFrame' object has no attribute 'dtype'
xgb_model.fit(X_train_scaled.values, y_train.values)
# ------------------------------------------------------------------

print(f"Rácio de Ponderação de Entorse (scale_pos_weight): {scale_ratio:.2f}")
print("--- ✅ FASE 3 CONCLUÍDA! MODELO XGBOOST TREINADO. ---")


--- FASE 3: TREINO DO MODELO (XGBoost) ---
Rácio de Ponderação de Entorse (scale_pos_weight): 0.80
--- ✅ FASE 3 CONCLUÍDA! MODELO XGBOOST TREINADO. ---


In [26]:
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix
import time

print("\n--- FASE 4: AVALIAÇÃO FINAL NO CONJUNTO DE TESTE (XGBoost) ---")

# ⏱️ Início da medição do tempo
start_time = time.perf_counter()

# 1. Previsões
y_pred = xgb_model.predict(X_test_scaled.values)

# 2. Métricas de Avaliação
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

# ⏱️ Fim da medição do tempo
end_time = time.perf_counter()
execution_time_ms = (end_time - start_time) * 1000

# 3. Resultados
print(f"Precisão (Accuracy): {accuracy:.4f}")
print(f"Recall (Sensibilidade): {recall:.4f}")
print(f"F-Score: {f1:.4f}")
print(f"Tempo de Execução: {execution_time_ms:.2f} ms")

print("\nMatriz de Confusão:")
print(cm)

# 4. Cálculo dos Falsos Negativos (Erro Crítico)
FN = cm[1, 0]
print(f"\nFalsos Negativos (FN - Entorse real não prevista): {FN}")

# 5. Validação do Projeto
if FN <= 2:
    print("\n--- ✅ PROJETO CONCLUÍDO E VALIDADO (ALTO RECALL COM XGBOOST) ---")
else:
    print("\n--- ⚠️ RECALL BAIXO: É NECESSÁRIO OTIMIZAR O XGBOOST ---")



--- FASE 4: AVALIAÇÃO FINAL NO CONJUNTO DE TESTE (XGBoost) ---
Precisão (Accuracy): 0.6912
Recall (Sensibilidade): 0.6800
F-Score: 0.7083
Tempo de Execução: 14.60 ms

Matriz de Confusão:
[[43 18]
 [24 51]]

Falsos Negativos (FN - Entorse real não prevista): 24

--- ⚠️ RECALL BAIXO: É NECESSÁRIO OTIMIZAR O XGBOOST ---
